In [33]:
import re

In [38]:
with open('logs_hw4.txt', 'r', encoding='utf-8') as file:
    content = file.read()


In [39]:
import re


def parse_logs(text: str) -> list[dict]:
    log_pattern = (
        r"^(\d{4}-\d{2}-\d{2}\s\d{2}(?::\d{2}){2})"
        r"\s(INFO|WARN|ERROR)\s?(.*)$"
    )

    user_pattern = r"user=(\S+)"
    action_pattern = r"action=(\S+)"
    ip_extract_pattern = r"ip=(\S+)"
    card_extract_pattern = r"card=(\S+)"

    email_pattern = (
        r"^[A-Za-z0-9_%+-]+"
        r"(?:\.[A-Za-z0-9_%+-]+)*"
        r"@[A-Za-z0-9-]+"
        r"(?:\.[A-Za-z0-9-]+)+$"
    )

    ip_pattern = (
        r"^(?:[1-9]?\d|1\d{2}|2[0-4]\d|25[0-5])"
        r"(?:\.(?:[1-9]?\d|1\d{2}|2[0-4]\d|25[0-5])){3}$"
    )

    card_pattern = r"^\d{4}(?:-\d{4}){3}$"

    logs = re.findall(log_pattern, text, flags=re.MULTILINE)

    res = []

    for ts, level, tail in logs:
        items = {
            "ts": ts,
            "level": level
        }

        errors = []

        user_match = re.search(user_pattern, tail)
        action_match = re.search(action_pattern, tail)
        ip_match = re.search(ip_extract_pattern, tail)
        card_match = re.search(card_extract_pattern, tail)

        if user_match:
            user = user_match.group(1)
        else:
            user = None

        if action_match:
            action = action_match.group(1)
        else:
            action = None

        if ip_match:
            ip = ip_match.group(1)
        else:
            ip = None

        if card_match:
            card = card_match.group(1)
        else:
            card = None

        items["user"] = user
        items["action"] = action
        items["ip"] = ip

        if user is None:
            errors.append("missing_user")
        elif not re.fullmatch(email_pattern, user):
            errors.append("invalid_email")

        if action is None:
            errors.append("missing_action")

        if ip is None:
            errors.append("missing_ip")
        elif not re.fullmatch(ip_pattern, ip):
            errors.append("invalid_ip")

        if card is None:
            items["card"] = None
        elif not re.fullmatch(card_pattern, card):
            items["card"] = None
            errors.append("invalid_card")
        else:
            items["card"] = "****-****-****-" + card[-4:]

        items["errors"] = errors
        res.append(items)

    return res

In [36]:
parse_logs(content)

[{'ts': '2024-03-15 10:51:25',
  'level': 'WARN',
  'user': 'user+tag@gmail.com',
  'action': 'login',
  'ip': '203.0.113.42',
  'card': None,
  'errors': []},
 {'ts': '2024-13-01 12:00:00',
  'level': 'INFO',
  'user': 'ivan@mail.ru',
  'action': 'login',
  'ip': '1.1.1.1',
  'card': None,
  'errors': []},
 {'ts': '2024-03-15 09:36:54',
  'level': 'WARN',
  'user': 'petr.sidorov@yandex.ru',
  'action': 'retry',
  'ip': '999.1.1.1',
  'card': None,
  'errors': ['invalid_ip']},
 {'ts': '2024-03-15 09:07:09',
  'level': 'ERROR',
  'user': 'user@domain.',
  'action': 'delete',
  'ip': '0.0.0.0',
  'card': None,
  'errors': ['invalid_email']},
 {'ts': '2024-03-15 10:28:11',
  'level': 'ERROR',
  'user': '@onlydomain.com',
  'action': 'pay',
  'ip': '300.300.300.300',
  'card': None,
  'errors': ['invalid_email', 'invalid_ip', 'invalid_card']},
 {'ts': '2024-03-15 10:04:06',
  'level': 'ERROR',
  'user': 'user+tag@gmail.com',
  'action': 'pay',
  'ip': '127.0.0.1',
  'card': None,
  'errors